# Deliverable 3 – Kampsimulator

INF201, NMBU

Amalie Berntsen og Sander Vikse

GitHub: https://github.com/amaliebern-code/inf201--deliverable-3

Vi har bygget videre på `Character`-klassen fra forelesning 8 og laget en enkel kampsimulator. Koden ligger i fire filer: `character.py`, `dice.py`, `items.py` og `battle.py`. Først viser vi hva hver klasse kan, og så kjører vi noen kamper.

In [1]:
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon
from battle import Battle

## 1. Character

En `Character` har navn, HP, maks HP, skade og eventuelt et våpen. Vi har lagt til `take_damage()`, `heal()` og `is_alive`, og endret `attack()` slik at den kaller `take_damage()` hos motstanderen i stedet for å endre `_hp` direkte.

In [2]:
hero = Character("Hero")
print(hero)
print("Navn:", hero.name)
print("HP:", hero.hp)
print("Maks HP:", hero.hp_max)
print("Skade:", hero.damage)
print("Våpen:", hero.weapon)
print("Lever:", hero.is_alive)

Character: Hero with 100 HP and 10 damage
Navn: Hero
HP: 100
Maks HP: 100
Skade: 10
Våpen: None
Lever: True


HP kan ikke gå under 0 eller over maks:

In [3]:
hero.take_damage(30)
print(hero.hp)

hero.heal(500)
print(hero.hp)

hero.take_damage(500)
print(hero.hp)
print(hero.is_alive)

70
100
0
False


Et angrep uten våpen gjør 10 i skade:

In [4]:
attacker = Character("Attacker")
target = Character("Target")

attacker.attack(target)
print(target.hp)

90


## 2. Dice

En terning med valgfritt antall sider. Standard er 6. Den kan kastes med `roll()` eller ved å kalle objektet direkte.

In [5]:
d6 = Dice()
d20 = Dice(sides=20)

print(d6)
print(d20)
print(d20.roll())
print(d20())

Dice(sides=6)
Dice(sides=20)
19
4


## 3. Weapon

Et våpen har et navn og en terning. Terningen bestemmer hvor mye skade våpenet gjør.

In [6]:
sword = Weapon("Sword", Dice(sides=12))

print(sword.name)
print(sword.damage_dice)
print(sword.roll_damage())
print(sword.roll_damage())
print(sword.roll_damage())

Sword
Dice(sides=12)
1
3
3


En karakter med våpen bruker terningkastet som skade. Uten våpen er skaden alltid 10.

In [7]:
armed = Character("Armed", weapon=sword)
unarmed = Character("Unarmed")
target = Character("Target")

armed.attack(target)
print(target.hp)

unarmed.attack(target)
print(target.hp)

88
78


## 4. Warrior og Mage

Begge arver fra `Character`.

`Warrior` har rustning (`armor`) som trekkes fra skaden den får:

In [8]:
warrior = Warrior("Warrior")
print("Rustning:", warrior.armor)

warrior.take_damage(20)
print(warrior.hp)

Rustning: 5
85


`Mage` har mana. Med minst 10 mana kaster den en trylleformel som gjør dobbel skade og koster 10 mana. Ellers gjør den et vanlig angrep og får tilbake 5 mana.

In [9]:
mage = Mage("Mage")
target = Character("Target")

for i in range(6):
    mage.attack(target)
    print("HP hos target:", target.hp, " Mana:", mage.mana)

HP hos target: 80  Mana: 20
HP hos target: 60  Mana: 10
HP hos target: 40  Mana: 0
HP hos target: 30  Mana: 5
HP hos target: 20  Mana: 10
HP hos target: 0  Mana: 0


De tre første angrepene gjør 20 i skade. Så er manaen brukt opp, og magikeren må gjøre vanlige angrep på 10 til den har spart opp nok til en ny trylleformel.

## 5. Battle

`Battle` tar inn to karakterer. `play_round()` spiller én runde, og `simulate_battle()` fortsetter til én av dem er død.

In [10]:
battle = Battle(Character("Hero"), Character("Enemy"))
battle.play_round()
battle.play_round()

Round 1: Hero 90 HP, Enemy 90 HP
Round 2: Hero 80 HP, Enemy 80 HP


## 6. Kamper

### Uten våpen

In [11]:
battle = Battle(Character("Hero"), Character("Enemy"))
winner = battle.simulate_battle()

Round 1: Hero 90 HP, Enemy 90 HP
Round 2: Hero 80 HP, Enemy 80 HP
Round 3: Hero 70 HP, Enemy 70 HP
Round 4: Hero 60 HP, Enemy 60 HP
Round 5: Hero 50 HP, Enemy 50 HP
Round 6: Hero 40 HP, Enemy 40 HP
Round 7: Hero 30 HP, Enemy 30 HP
Round 8: Hero 20 HP, Enemy 20 HP
Round 9: Hero 10 HP, Enemy 10 HP
Round 10: Hero 10 HP, Enemy 0 HP
Hero wins after 10 rounds!


Begge gjør 10 i skade hver runde. Helten slår først og vinner derfor alltid i runde 10.

### Med og uten våpen

Her har fienden et sverd med en 12-sidet terning, mens helten slår uten våpen.

In [12]:
hero = Character("Hero")
enemy = Character("Enemy", weapon=Weapon("Sword", Dice(sides=12)))

battle = Battle(hero, enemy)
winner = battle.simulate_battle()

Round 1: Hero 93 HP, Enemy 90 HP
Round 2: Hero 91 HP, Enemy 80 HP
Round 3: Hero 82 HP, Enemy 70 HP
Round 4: Hero 79 HP, Enemy 60 HP
Round 5: Hero 76 HP, Enemy 50 HP
Round 6: Hero 74 HP, Enemy 40 HP
Round 7: Hero 63 HP, Enemy 30 HP
Round 8: Hero 51 HP, Enemy 20 HP
Round 9: Hero 43 HP, Enemy 10 HP
Round 10: Hero 43 HP, Enemy 0 HP
Hero wins after 10 rounds!


Helten vinner selv om den ikke har våpen. Sverdet gjør mellom 1 og 12 i skade, som i snitt er 6,5, og det er mindre enn de 10 helten gjør hver gang. Et våpen gjør altså ikke nødvendigvis karakteren sterkere.

### Kriger mot magiker uten våpen

In [13]:
battle = Battle(Warrior("Warrior"), Mage("Mage"))
winner = battle.simulate_battle()

Round 1: Warrior 85 HP, Mage 90 HP
Round 2: Warrior 70 HP, Mage 80 HP
Round 3: Warrior 55 HP, Mage 70 HP
Round 4: Warrior 50 HP, Mage 60 HP
Round 5: Warrior 45 HP, Mage 50 HP
Round 6: Warrior 30 HP, Mage 40 HP
Round 7: Warrior 25 HP, Mage 30 HP
Round 8: Warrior 20 HP, Mage 20 HP
Round 9: Warrior 5 HP, Mage 10 HP
Round 10: Warrior 5 HP, Mage 0 HP
Warrior wins after 10 rounds!


Krigeren mister 15 HP i hver av de tre første rundene, fordi magikeren kaster trylleformler på 20 og rustningen tar 5. Etterpå har magikeren lite mana og gjør stort sett bare 5 i skade. Krigeren vinner med 5 HP igjen, og siden ingen har våpen, blir resultatet det samme hver gang.

### Kriger mot magiker med våpen

In [14]:
warrior = Warrior("Warrior", weapon=Weapon("Sword", Dice(sides=12)))
mage = Mage("Mage", weapon=Weapon("Staff", Dice(sides=6)))

battle = Battle(warrior, mage)
winner = battle.simulate_battle()

Round 1: Warrior 85 HP, Mage 90 HP
Round 2: Warrior 70 HP, Mage 82 HP
Round 3: Warrior 55 HP, Mage 79 HP
Round 4: Warrior 55 HP, Mage 68 HP
Round 5: Warrior 55 HP, Mage 58 HP
Round 6: Warrior 40 HP, Mage 49 HP
Round 7: Warrior 40 HP, Mage 40 HP
Round 8: Warrior 40 HP, Mage 36 HP
Round 9: Warrior 25 HP, Mage 24 HP
Round 10: Warrior 24 HP, Mage 13 HP
Round 11: Warrior 24 HP, Mage 9 HP
Round 12: Warrior 24 HP, Mage 0 HP
Warrior wins after 12 rounds!


Nå bruker begge terninger, så resultatet varierer. Vi kjørte cellen flere ganger, og både krigeren og magikeren vant noen av gangene. Kampen varer lenger enn uten våpen, fordi sverdet gjør mindre skade i snitt enn 10. Staven til magikeren har nesten ingenting å si, siden trylleformlene ikke bruker våpen og rustningen stopper det meste av et stavslag.

---
---
## Detaljert logg over AI-interaksjoner

Nedenfor er en kronologisk gjennomgang av de konkrete spørsmålene som ble stilt undervegs i utviklingen, og svarene/veiledningen som ble gitt:

### Task 2: Dice & `__repr__`
* **Spørsmål:** *Hva skrives ut hvis jeg kjører `print(Dice())` uten `__repr__`, og hva vil jeg at det skal stå?*
  * **Svar/Veiledning:** Forklarte at Python uten `__repr__` skriver ut objektets minneadresse (f.eks. `<__main__.Dice object at 0x... >`), og at oppgaven ber om at den skal returnere en lesbar streng som `Dice(sides=6)`.

### Task 3: Items & Komposisjon
* **Spørsmål:** *Hva eier hva? Har karakteren en terning, eller har våpenet det?*
  * **Svar/Veiledning:** Avklarte at våpenet eier terningen (`damage_dice`), mens karakteren eier våpenet (`weapon`).
* **Spørsmål:** *Hvordan får en fil tak i en klasse som ligger i en annen fil?*
  * **Svar/Veiledning:** Gjennomgikk Pythons `import`-system (f.eks. `from items import Dice`).
* **Spørsmål:** *Hvordan gjør jeg en parameter valgfri, og hvilken verdi betyr "ingenting"?*
  * **Svar/Veiledning:** Viste bruk av standardverdi `weapon = None` i `__init__`.
* **Spørsmål:** *Hvilke to tilfeller må `attack()` skille mellom nå, og hvordan sjekker jeg hvilket som gjelder?*
  * **Svar/Veiledning:** Forklarte logikken for å sjekke `if self.weapon is not None:` (rulle våpenskade) vs. `else:` (bruke karakterens faste basesskade).

### Task 4: Spesialisering & Arv
* **Spørsmål:** *Hva er likt med `Character`, og hva er annerledes? (Bare det som er annerledes, skal skrives.)*
  * **Svar/Veiledning:** Identifiserte at `Warrior` kun overstyrer/legger til `armor` og `take_damage()`, mens `Mage` overstyrer/legger til `mana`, `mana_max` og `attack()`.
* **Spørsmål:** *Hvordan sier jeg at en klasse arver fra en annen?*
  * **Svar/Veiledning:** Viste syntaks for arv: `class Warrior(Character):`.
* **Spørsmål:** *Hvordan får jeg den nye klassen til å gjøre alt `Character.__init__` gjør, og så legge til sitt eget?*
  * **Svar/Veiledning:** Forklarte bruk av `super().__init__(...)` for å gjenbruke konstruktøren til foreldreklassen.
* **Spørsmål:** *Hvilken metode må krigeren endre for at rustningen skal virke, og hva skjer hvis skaden er mindre enn rustningen?*
  * **Svar/Veiledning:** Forklarte at `take_damage()` må overstyres, og hvordan `max(0, damage - armor)` forhindrer negativ skade (at krigeren heales).
* **Spørsmål:** *Hvordan kaller jeg den opprinnelige versjonen av en metode fra en overstyrt metode?*
  * **Svar/Veiledning:** Viste bruk av `super().metodenavn(...)` i `Warrior.take_damage()` og `Mage.attack()`.
* **Spørsmål:** *Hvilke to situasjoner kan magikeren være i når den angriper, og hva skjer med manaen i hver av dem?*
  * **Svar/Veiledning:** Avklarte vilkårene for magisk angrep ($\ge 10$ mana, trekker 10 mana, dobbel skade uten våpen) vs. vanlig angrep ($< 10$ mana, genererer 5 mana, bruker base/våpenskade).

### Task 5: Running Battles & Arkitektur
* **Spørsmål:** *Hva må en kamp vite om for å kunne kjøre?*
  * **Svar/Veiledning:** Avklarte at `Battle.__init__` tar imot to parametere: `hero` og `enemy`.
* **Spørsmål:** *Hva skjer i én runde, i hvilken rekkefølge, og når skal fienden ikke få slå tilbake?*
  * **Svar/Veiledning:** Avklarte at `hero` angriper først, og at `enemy` kun angriper tilbake dersom `enemy.is_alive` fortsatt er `True`.
* **Spørsmål:** *Hva er betingelsen for at kampen skal fortsette?*
  * **Svar/Veiledning:** Forklarte `while self.hero.is_alive and self.enemy.is_alive:`.
* **Spørsmål:** *Kan løkken noen gang gå i det uendelige, og hvordan stopper jeg det?*
  * **Svar/Veiledning:** Diskuterte tilfeller der kamper kjører fast (f.eks. høy armor), og løste det ved å innføre en `max_rounds`-teller.
* **Spørsmål:** *Hvordan finner jeg ut hvem som vant når løkken er ferdig?*
  * **Svar/Veiledning:** Viste logikk for tilstandssjekk etter løkken (`is_alive` på helt vs. fiende) for å kåre vinner eller uavgjort.
* **Spørsmål:** *Hva er den lureste måten å samle alt sammen i en notebook?*
  * **Svar/Veiledning:** Anbefalte modulær struktur med `.py`-filer for klassene, bruk av `%load_ext autoreload` i Jupyter, og oppdeling i ryddige testceller.
---